# Preprocessing (Amazon Review Data 2018)

Builds `<dataset>_review.parquet` and `<dataset>_meta.parquet` for one category:
5-core review file, recursive 5-core filtering, random sample of users up to about 50,000 interactions
(seed 42), and item metadata (`title. category. description`) aligned to the item ids.
Note that sampling users does not preserve the 5-core property on items.

In [ ]:
import gzip
import json

import numpy as np
import pandas as pd
from tqdm import tqdm

CATEGORY = 'Movies_and_TV'   # Movies_and_TV | CDs_and_Vinyl | Electronics
OUT_NAME = {'Movies_and_TV': 'Movies_and_TV', 'CDs_and_Vinyl': 'CD_Vinyl', 'Electronics': 'electronics'}[CATEGORY]

In [ ]:
url_reviews = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/categoryFilesSmall/{CATEGORY}_5.json.gz"
url_meta = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/metaFiles2/meta_{CATEGORY}.json.gz"

!wget -nc "$url_reviews"
!wget -nc "$url_meta"

In [ ]:
review_path = f"{CATEGORY}_5.json.gz"

chunks = []
with pd.read_json(review_path, lines=True, chunksize=100000) as reader:
    for chunk in tqdm(reader):
        chunks.append(chunk)
df_rev = pd.concat(chunks)

df = df_rev[['reviewerID', 'asin', 'reviewText', 'overall', 'unixReviewTime', 'summary']]
df = df.rename(columns={'reviewerID': 'user_id', 'overall': 'rating', 'unixReviewTime': 'timestamp'})
X = df.drop_duplicates(subset=['user_id', 'asin'], keep='last')

In [ ]:
def k_core_filter(df, k=5):
    while True:
        user_counts = df['user_id'].value_counts()
        item_counts = df['asin'].value_counts()

        valid_users = user_counts[user_counts >= k].index
        valid_items = item_counts[item_counts >= k].index

        new_df = df[(df['user_id'].isin(valid_users)) & (df['asin'].isin(valid_items))]

        if len(new_df) == len(df):
            break
        df = new_df

    return df

print(f"Interactions before k-core: {len(X)}")
df_kcore = k_core_filter(X, k=5)
print(f"Interactions after 5-core: {len(df_kcore)}")

In [ ]:
user_ids = df_kcore['user_id'].unique()
user_map = {id_: i for i, id_ in enumerate(user_ids)}
df_kcore['user_id_int'] = df_kcore['user_id'].map(user_map)

item_ids = df_kcore['asin'].unique()
item_map = {id_: i for i, id_ in enumerate(item_ids)}
df_kcore['item_id_int'] = df_kcore['asin'].map(item_map)

print(f"Users: {len(user_map)}")
print(f"Items: {len(item_map)}")

In [ ]:
df_final = df_kcore.sort_values(by=['user_id_int', 'timestamp'])

In [ ]:
valid_asins = set(df_final['asin'].unique())
print(f"Looking up metadata for {len(valid_asins)} items.")

meta_path = f"meta_{CATEGORY}.json.gz"
print(f"meta_path: {meta_path}")

data = []
with gzip.open(meta_path, 'r') as f:
    for line in tqdm(f):
        try:
            d = json.loads(line.strip())

            if d['asin'] in valid_asins:
                data.append({
                    'asin': d['asin'],
                    'title': d.get('title', ''),
                    'description': d.get('description', ''),
                    'category': d.get('category', [])
                })
        except ValueError:
            continue

df_meta = pd.DataFrame(data)

def clean_text(row):
    title = row['title']
    desc = row['description']
    cats = row['category']

    if isinstance(desc, list): desc = " ".join(desc)
    if isinstance(cats, list): cats = " ".join(cats)

    return f"{title}. {cats}. {desc}"

df_meta['meta_text'] = df_meta.apply(clean_text, axis=1)

asin_to_int = dict(zip(df_final['asin'], df_final['item_id_int']))
df_meta['item_id_int'] = df_meta['asin'].map(asin_to_int)

df_meta = df_meta.sort_values('item_id_int').reset_index(drop=True)

df_meta.head()

In [ ]:
np.random.seed(42)

df_kcore = df_final.copy()

print(
    "Before sampling:",
    len(df_kcore),
    df_kcore['user_id_int'].nunique(),
    df_kcore['item_id_int'].nunique()
)

user_counts = df_kcore['user_id_int'].value_counts()

avg_reviews = user_counts.mean()

num_users = int(50000 / avg_reviews)

sampled_ids = np.random.choice(
    user_counts.index,
    size=num_users,
    replace=False
)

df_sampled = df_kcore[
    df_kcore['user_id_int'].isin(sampled_ids)
].copy()

print(
    "After sampling:",
    len(df_sampled),
    df_sampled['user_id_int'].nunique(),
    df_sampled['item_id_int'].nunique()
)

df_sampled.reset_index(drop=True, inplace=True)

unique_users = sorted(df_sampled['user_id_int'].unique())
user_map = {old: new for new, old in enumerate(unique_users)}

df_sampled['user_id_int'] = df_sampled['user_id_int'].map(user_map)

unique_items = sorted(df_sampled['item_id_int'].unique())
item_map = {old: new for new, old in enumerate(unique_items)}

df_sampled['item_id_int'] = df_sampled['item_id_int'].map(item_map)

print(
    f"Final dataset: {len(df_sampled)} interactions, "
    f"{df_sampled['user_id_int'].nunique()} users, "
    f"{df_sampled['item_id_int'].nunique()} items"
)
df_meta_unique = df_meta.drop_duplicates(subset=['asin'])

unique_items_df = (
    df_sampled[['item_id_int', 'asin']]
    .drop_duplicates()
    .sort_values('item_id_int')
)

df_meta_aligned = pd.merge(
    unique_items_df,
    df_meta_unique[['asin', 'meta_text']],
    on='asin',
    how='left'
)

df_meta_aligned['meta_text'] = df_meta_aligned[
    'meta_text'
].fillna("Product description unavailable")

print(
    f"Aligned metadata: {len(df_meta_aligned)} rows"
)

assert len(df_meta_aligned) == len(unique_items_df)

In [ ]:
df_sampled.to_parquet(f'{OUT_NAME}_review.parquet')
df_meta_aligned.to_parquet(f'{OUT_NAME}_meta.parquet')